In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler, Dataset
import numpy as np
import random
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve,
                             balanced_accuracy_score, matthews_corrcoef, precision_recall_curve,
                             average_precision_score)
import seaborn as sns
import csv
from torchvision.transforms import functional as F

# For Google Colab: Mount Google Drive
try:
    from google.colab import drive
    DRIVE_MOUNTED = False
except ImportError:
    DRIVE_MOUNTED = False

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Define the device for computation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Mount Google Drive to save and load the model
if DRIVE_MOUNTED is False and 'google.colab' in sys.modules:
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
else:
    print("Google Drive not mounted or not running in Google Colab.")

# Define the folder to save model checkpoints
date_str = datetime.now().strftime('%Y%m%d')
checkpoint_folder = f'/content/drive/My Drive/FYP/VGGModel/HQ2ltst_{date_str}/' if DRIVE_MOUNTED else './checkpoints/'
os.makedirs(checkpoint_folder, exist_ok=True)

# Data Augmentation for Training Set
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to 224x224
    transforms.RandomHorizontalFlip(),  # Flip horizontally
    transforms.RandomRotation(15),  # Random rotation
    transforms.RandomCrop(224, padding=10),  # Crop with padding
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # Normalize
])

# Simple resizing for validation and test sets
transform_val_test = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to 224x224
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # Normalize
])

# Load datasets
data_path = '/content/drive/My Drive/FYP/Dataset/HQ2/train' if DRIVE_MOUNTED else './data/train'
test_data_path = '/content/drive/My Drive/FYP/Dataset/HQ2/test' if DRIVE_MOUNTED else './data/test'

train_dataset_full = datasets.ImageFolder(data_path, transform=transform_train)
test_dataset = datasets.ImageFolder(test_data_path, transform=transform_val_test)

# Split the dataset into 80% training and 20% validation
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# Identify class counts
train_labels = [label for _, label in train_dataset]
label_counts = Counter(train_labels)
print(f"Class Counts: {label_counts}")

# Calculate class weights
total_samples = len(train_dataset)
num_classes = len(label_counts)
class_weights = [total_samples / (num_classes * label_counts[i]) for i in range(num_classes)]
print(f"Class Weights: {class_weights}")

# Convert class weights to tensor and move to device
class_weights_tensor = torch.tensor(class_weights).to(device)

# Assign weight to each sample for sampler
samples_weights = [class_weights[label] for label in train_labels]

# Create WeightedRandomSampler
sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Use Pre-trained VGG-16 model and modify it for binary classification
class VGG16Modified(nn.Module):
    def __init__(self):
        super(VGG16Modified, self).__init__()
        # Load pre-trained VGG-16 model
        self.vgg = models.vgg16(weights='IMAGENET1K_V1')
        # Replace the classifier with a custom binary classification layer
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_ftrs, 2)  # Binary classification
        )

    def forward(self, x):
        return self.vgg(x)

# Initialize the VGG model
model = VGG16Modified().to(device)

# Define loss function and optimizer with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

# Training function
def train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=20):
    best_model_path = os.path.join(checkpoint_folder, f"best_model_vgg_{date_str}.pth")
    best_accuracy = 0

    for epoch in range(epochs):
        # Training loop
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = correct / total
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {total_loss:.4f}, Train Accuracy: {train_acc:.4f}")

        # Validation loop
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

        # Save the best model
        if val_acc > best_accuracy:
            best_accuracy = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"Best model saved at epoch {epoch+1} with accuracy {val_acc:.4f}")

        scheduler.step(val_acc)

# Start training
train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=10)

# Test function
def test(model, test_loader):
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_accuracy = test_correct / test_total
    print(f"Test Accuracy: {test_accuracy:.4f}")

# Test the model
test(model, test_loader)


Using device: cpu


KeyboardInterrupt: 

In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler, Dataset
import numpy as np
import random
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve,
                             balanced_accuracy_score, matthews_corrcoef, precision_recall_curve,
                             average_precision_score)
import seaborn as sns
import csv
from torchvision.transforms import functional as F

# For Google Colab: Mount Google Drive
try:
    from google.colab import drive
    DRIVE_MOUNTED = False
except ImportError:
    DRIVE_MOUNTED = False

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Define the device for computation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Mount Google Drive to save and load the model
if DRIVE_MOUNTED is False and 'google.colab' in sys.modules:
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
else:
    print("Google Drive not mounted or not running in Google Colab.")

# Define the folder to save model checkpoints
date_str = datetime.now().strftime('%Y%m%d')
checkpoint_folder = f'/content/drive/My Drive/FYP/VGGModel/HQ2ltst_{date_str}/'
os.makedirs(checkpoint_folder, exist_ok=True)

# Data Augmentation for Training Set
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to 224x224
    transforms.RandomHorizontalFlip(),  # Flip horizontally
    transforms.RandomRotation(15),  # Random rotation
    transforms.RandomCrop(224, padding=10),  # Crop with padding
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # Normalize
])

# Simple resizing for validation and test sets
transform_val_test = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to 224x224
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # Normalize
])

# Define paths
data_path = '/content/drive/My Drive/FYP/Dataset/HQ2/train' if DRIVE_MOUNTED else './data/train'
test_data_path = '/content/drive/My Drive/FYP/Dataset/HQ2/test' if DRIVE_MOUNTED else './data/test'

# Load datasets
train_dataset_full = datasets.ImageFolder(data_path, transform=transform_train)
test_dataset = datasets.ImageFolder(test_data_path, transform=transform_val_test)

# Split the dataset into training and validation sets
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# Identify class counts
train_labels = [label for _, label in train_dataset]
label_counts = Counter(train_labels)
print(f"Class Counts: {label_counts}")

# Function to display original and transformed images
def visualize_transformations(dataset, transform, num_samples=4):
    """Visualize original and transformed images."""
    plt.figure(figsize=(12, 6))
    for i in range(num_samples):
        idx = random.randint(0, len(dataset) - 1)
        image, label = dataset[idx]

        # Original image
        original_image = image if isinstance(image, torch.Tensor) else F.to_tensor(image)
        original_image = original_image.numpy().transpose((1, 2, 0))
        original_image = np.clip(original_image * np.array([0.229, 0.224, 0.225]) +
                                 np.array([0.485, 0.456, 0.406]), 0, 1)

        # Transformed image
        transformed_image = transform(image) if transform else image
        transformed_image = transformed_image.numpy().transpose((1, 2, 0))
        transformed_image = np.clip(transformed_image * np.array([0.229, 0.224, 0.225]) +
                                    np.array([0.485, 0.456, 0.406]), 0, 1)

        # Plot original and transformed images
        plt.subplot(2, num_samples, i + 1)
        plt.imshow(original_image)
        plt.title(f"Original (Label {label})")
        plt.axis('off')

        plt.subplot(2, num_samples, i + 1 + num_samples)
        plt.imshow(transformed_image)
        plt.title(f"Transformed (Label {label})")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Visualize samples from the training dataset
print("Visualizing Original and Transformed Images:")
visualize_transformations(train_dataset_full, transform_train, num_samples=4)
# Calculate class weights
total_samples = len(train_dataset)
num_classes = len(label_counts)
class_weights = [total_samples / (num_classes * label_counts[i]) for i in range(num_classes)]
print(f"Class Weights: {class_weights}")

# Convert class weights to tensor and move to device
class_weights_tensor = torch.tensor(class_weights).to(device)

# Assign weight to each sample for sampler
samples_weights = [class_weights[label] for label in train_labels]

# Create WeightedRandomSampler
sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)

from torchvision.transforms import functional as F

class CustomImageDataset(Dataset):
    def __init__(self, dataset, transform_major, transform_minority, minority_class):
        self.dataset = dataset
        self.transform_major = transform_major
        self.transform_minority = transform_minority
        self.minority_class = minority_class

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if label == self.minority_class:
            # Convert the tensor to a PIL image before applying the transformation
            image = F.to_pil_image(image)
            image = self.transform_minority(image)
        else:
            image = F.to_pil_image(image)  # Convert tensor to PIL image
            image = self.transform_major(image)
        return image, label

# Determine minority and majority classes based on class counts
# Assuming class with fewer samples is minority
minority_class = min(label_counts, key=label_counts.get)
majority_class = max(label_counts, key=label_counts.get)
print(f"Minority Class: {minority_class}, Majority Class: {majority_class}")

# Create custom training dataset
custom_train_dataset = CustomImageDataset(
    train_dataset,
    transform_major=transform_train,
    transform_minority=transform_minority,
    minority_class=minority_class
)

# Update train_labels and sampler for the custom dataset
custom_train_labels = [label for _, label in custom_train_dataset]
samples_weights = [class_weights[label] for label in custom_train_labels]

# Create WeightedRandomSampler for the custom dataset
sampler = WeightedRandomSampler(weights=samples_weights, num_samples=len(samples_weights), replacement=True)

# Create data loaders
batch_size = 32
train_loader = DataLoader(custom_train_dataset, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Use Pre-trained VGG-16 model and modify it for binary classification
class VGG16Modified(nn.Module):
    def __init__(self):
        super(VGG16Modified, self).__init__()
        # Load pre-trained VGG-16 model
        self.vgg = models.vgg16(pretrained=True)
        # Freeze early layers if desired
        # for param in self.vgg.features.parameters():
        #     param.requires_grad = False
        # Replace the classifier with a custom binary classification layer
        num_ftrs = self.vgg.classifier[6].in_features
        self.vgg.classifier[6] = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_ftrs, num_classes)  # num_classes = 2 for binary classification
        )

    def forward(self, x):
        return self.vgg(x)

# Initialize the VGG model
num_classes = 2  # Binary classification
model = VGG16Modified().to(device)

# Define loss function and optimizer with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

def validate(model, loader, criterion):
    model.eval()
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, predicted = torch.max(outputs.data, 1)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # Probability of class 1

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    accuracy = 100 * correct / total
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    mcc = matthews_corrcoef(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs)
    ap_score = average_precision_score(all_labels, all_probs)

    cls_report = classification_report(all_labels, all_preds, target_names=['Class 0', 'Class 1'])
    conf_matrix = confusion_matrix(all_labels, all_preds)

    precision, recall, _ = precision_recall_curve(all_labels, all_probs)

    print(f'Validation Accuracy: {accuracy:.2f}%')
    print(f'Balanced Accuracy: {balanced_acc:.2f}%')
    print(f'Matthews Correlation Coefficient: {mcc:.2f}')
    print(f'ROC-AUC: {roc_auc:.2f}')
    print(f'Average Precision (AP): {ap_score:.2f}')
    print("Classification Report:")
    print(cls_report)
    print("Confusion Matrix:")
    print(conf_matrix)

    # Plot Confusion Matrix
    plt.figure(figsize=(6,5))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Class 0', 'Class 1'],
                yticklabels=['Class 0', 'Class 1'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

    # Plot ROC Curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc='lower right')
    plt.show()

    # Plot Precision-Recall Curve
    plt.figure(figsize=(6,5))
    plt.plot(recall, precision, label=f'AP = {ap_score:.2f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall (PR) Curve')
    plt.legend(loc='lower left')
    plt.grid(True)
    plt.show()

    return accuracy, balanced_acc, mcc, roc_auc, ap_score

def train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=20):
    best_roc_auc = 0.0
    train_losses = []
    val_accuracies = []
    val_balanced_accuracies = []
    val_mccs = []
    val_roc_aucs = []
    val_ap_scores = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)  # Gradient clipping
            optimizer.step()

            running_loss += loss.item()

        # Calculate average training loss
        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.6f}")

        # Validate the model and get all metrics
        val_acc, val_balanced_acc, val_mcc, val_roc_auc, val_ap_score = validate(model, val_loader, criterion)
        val_accuracies.append(val_acc)
        val_balanced_accuracies.append(val_balanced_acc)
        val_mccs.append(val_mcc)
        val_roc_aucs.append(val_roc_auc)
        val_ap_scores.append(val_ap_score)
        scheduler.step(val_acc)

        # Save the best model based on ROC-AUC
        if val_roc_auc > best_roc_auc:
            best_roc_auc = val_roc_auc
            best_model_path = os.path.join(checkpoint_folder, f"best_model_vgg_{date_str}.pth")
            torch.save(model.state_dict(), best_model_path)
            print(f"New best model saved with ROC-AUC: {best_roc_auc:.2f} at {best_model_path}")

        # Save the current model as the final model (replacing the previous final model)
        final_model_path = os.path.join(checkpoint_folder, f"final_model_vgg_{date_str}.pth")
        torch.save(model.state_dict(), final_model_path)
        print(f"Final model saved at {final_model_path} (replacing the previous one)")

    # Plot training loss and validation metrics
    plt.figure(figsize=(25, 5))

    # Training Loss
    plt.subplot(1, 5, 1)
    plt.plot(train_losses, label='Training Loss', color='blue')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.grid(True)
    plt.legend()

    # Validation Accuracy
    plt.subplot(1, 5, 2)
    plt.plot(val_accuracies, label='Validation Accuracy', color='green')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.title('Validation Accuracy')
    plt.grid(True)
    plt.legend()

    # Validation Balanced Accuracy
    plt.subplot(1, 5, 3)
    plt.plot(val_balanced_accuracies, label='Validation Balanced Accuracy', color='orange')
    plt.xlabel('Epochs')
    plt.ylabel('Balanced Accuracy (%)')
    plt.title('Validation Balanced Accuracy')
    plt.grid(True)
    plt.legend()

    # Validation MCC
    plt.subplot(1, 5, 4)
    plt.plot(val_mccs, label='Validation MCC', color='purple')
    plt.xlabel('Epochs')
    plt.ylabel('MCC')
    plt.title('Validation MCC')
    plt.grid(True)
    plt.legend()

    # Validation ROC-AUC and AP
    plt.subplot(1, 5, 5)
    plt.plot(val_roc_aucs, label='Validation ROC-AUC', color='red')
    plt.plot(val_ap_scores, label='Validation AP', color='cyan')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.title('Validation ROC-AUC & AP')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

    # Optionally, save metrics to a CSV file
    save_metrics(train_losses, val_accuracies, val_balanced_accuracies, val_mccs, val_roc_aucs, val_ap_scores)



def test(model, loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # Probability of class 1

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    accuracy = 100 * correct / total
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    mcc = matthews_corrcoef(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs)
    ap_score = average_precision_score(all_labels, all_probs)

    cls_report = classification_report(all_labels, all_preds, target_names=['Class 0', 'Class 1'])
    conf_matrix = confusion_matrix(all_labels, all_preds)

    precision, recall, _ = precision_recall_curve(all_labels, all_probs)

    print(f'Test Loss: {test_loss/len(loader):.6f}, Test Accuracy: {accuracy:.2f}%')
    print(f'Balanced Accuracy: {balanced_acc:.2f}%')
    print(f'Matthews Correlation Coefficient: {mcc:.2f}')
    print(f'Test ROC-AUC: {roc_auc:.2f}')
    print(f'Average Precision (AP): {ap_score:.2f}')
    print("Test Classification Report:")
    print(cls_report)
    print("Test Confusion Matrix:")
    print(conf_matrix)

    # Plot Confusion Matrix
    plt.figure(figsize=(6,5))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Greens',
                xticklabels=['Class 0', 'Class 1'],
                yticklabels=['Class 0', 'Class 1'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Test Confusion Matrix')
    plt.show()

    # Plot ROC Curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})', color='purple')
    plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc='lower right')
    plt.show()

    # Plot Precision-Recall Curve
    plt.figure(figsize=(6,5))
    plt.plot(recall, precision, label=f'AP = {ap_score:.2f}', color='cyan')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall (PR) Curve')
    plt.legend(loc='lower left')
    plt.grid(True)
    plt.show()

    return accuracy, balanced_acc, mcc, roc_auc, ap_score

def save_metrics(train_losses, val_accuracies, val_balanced_accuracies, val_mccs, val_roc_aucs, val_ap_scores):
    metrics_path = os.path.join(checkpoint_folder, 'metrics.csv')
    with open(metrics_path, 'w', newline='') as csvfile:
        fieldnames = ['Epoch', 'Training Loss', 'Validation Accuracy',
                      'Validation Balanced Accuracy', 'Validation MCC',
                      'Validation ROC-AUC', 'Validation AP']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        writer.writeheader()
        for epoch in range(len(train_losses)):
            writer.writerow({
                'Epoch': epoch + 1,
                'Training Loss': train_losses[epoch],
                'Validation Accuracy': val_accuracies[epoch],
                'Validation Balanced Accuracy': val_balanced_accuracies[epoch],
                'Validation MCC': val_mccs[epoch],
                'Validation ROC-AUC': val_roc_aucs[epoch],
                'Validation AP': val_ap_scores[epoch]
            })
    print(f"Metrics saved to {metrics_path}")


if __name__ == "__main__":
    # Visualize original and transformed images from the training dataset
    print("Visualizing Original and Transformed Images:")
    visualize_transformations(train_dataset_full, transform_train, num_samples=4)

    # Define the number of epochs
    epochs = 20  # Adjust the number of epochs as needed

    # Train the model
    train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=epochs)

    # Optionally, load the best model and perform testing
    best_model_path = os.path.join(checkpoint_folder, f"best_model_vgg_{date_str}.pth")
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        print(f"Loaded best model from {best_model_path}")
    else:
        print(f"Best model not found at {best_model_path}, using the final model.")

    # Test the model
    test(model, test_loader, criterion)

Using device: cpu
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Class Counts: Counter({1: 505, 0: 109})
Visualizing Original and Transformed Images:


TypeError: pic should be PIL Image or ndarray. Got <class 'torch.Tensor'>

<Figure size 1200x600 with 0 Axes>

In [6]:
# Section 3: Final Model Evaluation - AUC, ROC Curve, Precision-Recall Curve

# Get probability scores for ROC/AUC calculation
model.eval()
y_true = []
y_scores = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        probas = torch.softmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy())
        y_scores.extend(probas.cpu().numpy())

# AUC (Area Under Curve)
roc_auc = roc_auc_score(y_true, y_scores, multi_class='ovr')
print(f"ROC AUC: {roc_auc:.4f}")

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_scores[:, 1])
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_true, y_scores[:, 1])
plt.figure(figsize=(10, 6))
plt.plot(recall, precision, color='blue', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()

# Average Precision Score
avg_precision = average_precision_score(y_true, y_scores, average="macro")
print(f"Average Precision Score: {avg_precision:.4f}")


ValueError: y should be a 1d array, got an array of shape (30, 2) instead.